In [ ]:
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor
import numpy as np

In [ ]:
ds = load_dataset("sanchit-gandhi/gtzan")
ds = ds["train"].train_test_split(test_size=0.1, shuffle=True)
id2label_fn = ds["train"].features["genre"].int2str

In [ ]:
import gradio as gr


def generate_audio():
    example = ds["train"].shuffle()[0]
    audio = example["audio"]
    return (
        audio["sampling_rate"],
        audio["array"],
    ), id2label_fn(example["genre"])


with gr.Blocks() as demo:
    with gr.Column():
        for _ in range(4):
            audio, label = generate_audio()
            output = gr.Audio(audio, label=label)

demo.launch(debug=True)

In [ ]:
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True, return_attention_mask=True)
feature_extractor

In [ ]:
ds = ds.cast_column("audio", Audio(sampling_rate=feature_extractor.sampling_rate))

In [ ]:
sample = ds['train'][0]["audio"]
print(f"Mean: {np.mean(sample['array']):.3}, Variance: {np.var(sample['array']):.3}")

In [ ]:
inputs = feature_extractor(sample["array"], sampling_rate=sample["sampling_rate"])

print(f"inputs keys: {list(inputs.keys())}")

print(
    f"Mean: {np.mean(inputs['input_values']):.3}, Variance: {np.var(inputs['input_values']):.3}"
)

In [ ]:
inputs["input_values"]

In [ ]:
def preprocess_audio(examples):
    audio_arrays = [x['array'] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * 30.0),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

In [ ]:
ds_encoded = ds.map(
    preprocess_audio,
    remove_columns=["audio", "file"],
    batched=True,
    batch_size=100,
    num_proc=1
)

In [ ]:
ds_encoded = ds_encoded.rename_column("genre", "label")

In [ ]:
id2lable = {
    str(i): id2label_fn(i)
    for i in range(len(ds_encoded["train"].features["label"].names))
}
label2id = { v: k for k, v in id2lable.items()}

In [ ]:
len(id2lable)

In [ ]:
from transformers import AutoModelForAudioClassification

num_labels = len(id2lable)


model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2lable,
    ignore_mismatched_sizes=True
)

In [ ]:
ds_encoded["train"][0]

In [ ]:
from transformers import TrainingArguments


model_name = model_id.split("/")[-1]
batch_size = 2
gradient_accumulation_steps = 1
num_train_epochs = 10


training_args = TrainingArguments(
    f"{model_name}-finetuned-gtzan",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    warmup_steps=100,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
)

In [ ]:
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=ds_encoded["train"],
    eval_dataset=ds_encoded["test"],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()